# Descarga de datos históricos del EUR/USD

En este notebook se descargan y almacenan los datos diarios del par EUR/USD.

La fuente utilizada inicialmente es Yahoo Finance mediante la librería yfinance. 
Los datos se guardan sin transformar en la carpeta raw (en data) para conservar una copia reproducible de la información original

In [1]:
import json
from pathlib import Path

import pandas as pd
import yfinance as yf

# Detecta la carpeta principal del proyecto
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual

# Carpeta donde se guardarán los datos originales
ruta_raw = ruta_proyecto / "data" / "raw"
ruta_raw.mkdir(parents=True, exist_ok=True)

print("Carpeta actual:", ruta_actual)
print("Carpeta del proyecto:", ruta_proyecto)
print("Carpeta de datos:", ruta_raw)

Carpeta actual: C:\TFM_EURUSD\notebooks
Carpeta del proyecto: C:\TFM_EURUSD
Carpeta de datos: C:\TFM_EURUSD\data\raw


In [3]:
# Configuración de la descarga
TICKER = "EURUSD=X"
FECHA_INICIO = "2004-01-01"

# yfinance interpreta la fecha final como exclusiva
HOY_UTC = pd.Timestamp.now(tz="UTC").normalize()
FECHA_FIN = (HOY_UTC - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

INTERVALO = "1d"

print("Activo:", TICKER)
print("Fecha inicial:", FECHA_INICIO)
print("Fecha final exclusiva:", FECHA_FIN)
print("Intervalo:", INTERVALO)

Activo: EURUSD=X
Fecha inicial: 2004-01-01
Fecha final exclusiva: 2026-07-15
Intervalo: 1d


In [4]:
datos = yf.download(
    TICKER,
    start=FECHA_INICIO,
    end=FECHA_FIN,
    interval=INTERVALO,
    auto_adjust=False,
    progress=False,
    multi_level_index=False
)

if datos.empty:
    raise RuntimeError("No se descargaron datos. Revisa la conexión o el ticker.")

print("Descarga completada.")
print("Número de filas:", len(datos))

Descarga completada.
Número de filas: 5845


In [5]:
# Me aseguro que las fechas estén en el formato correcto

datos.index = pd.to_datetime(datos.index)
datos.index.name = "Date"

# Orden cronológico
datos = datos.sort_index()

# Columnas mínimas esperadas
columnas_esperadas = {"Open", "High", "Low", "Close"}
columnas_faltantes = columnas_esperadas.difference(datos.columns)

if columnas_faltantes:
    raise ValueError(
        f"No se encontraron estas columnas: {sorted(columnas_faltantes)}"
    )

if not datos.index.is_unique:
    raise ValueError("Existen fechas duplicadas en los datos.")

if not datos.index.is_monotonic_increasing:
    raise ValueError("Las fechas no están correctamente ordenadas.")

print("Validación correcta.")
print("Primera fecha:", datos.index.min().date())
print("Última fecha:", datos.index.max().date())
print("Número de jornadas:", len(datos))

Validación correcta.
Primera fecha: 2004-01-01
Última fecha: 2026-07-14
Número de jornadas: 5845


In [7]:
print("Primeras cinco jornadas:")
display(datos.head())
print("Últimas cinco jornadas:")
display(datos.tail())
print("Tipos de datos:")
display(datos.dtypes.to_frame(name="tipo"))

Primeras cinco jornadas:


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2004-01-01,1.258194,1.258194,1.260796,1.247396,1.259002,0
2004-01-02,1.258194,1.258194,1.262802,1.252693,1.258194,0
2004-01-05,1.268698,1.268698,1.269406,1.263695,1.263903,0
2004-01-06,1.272103,1.272103,1.280803,1.267202,1.268907,0
2004-01-07,1.264095,1.264095,1.273999,1.262499,1.272394,0


Últimas cinco jornadas:


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-07-08,1.140381,1.140381,1.143249,1.139186,1.140095,0
2026-07-09,1.142204,1.142204,1.145082,1.142126,1.142191,0
2026-07-10,1.143341,1.143341,1.146263,1.141839,1.143380,0
2026-07-13,1.140446,1.140446,1.144571,1.138680,1.140368,0
2026-07-14,1.138433,1.138433,1.146132,1.137877,1.138369,0


Tipos de datos:


,tipo
Adj Close,float64
Close,float64
High,float64
Low,float64
Open,float64
Volume,int64


In [8]:
valores_faltantes = datos.isna().sum().to_frame(
    name="valores_faltantes"
)

valores_faltantes["porcentaje"] = (
    valores_faltantes["valores_faltantes"] / len(datos) * 100
).round(2)

display(valores_faltantes)

print("Fechas duplicadas:", datos.index.duplicated().sum())

,valores_faltantes,porcentaje
Adj Close,0,0.0
Close,0,0.0
High,0,0.0
Low,0,0.0
Open,0,0.0
Volume,0,0.0


Fechas duplicadas: 0


In [9]:
resumen = pd.DataFrame({
    "valor": [
        TICKER,
        datos.index.min().date(),
        datos.index.max().date(),
        len(datos),
        datos.index.duplicated().sum(),
        int(datos.isna().sum().sum())
    ]
}, index=[
    "ticker",
    "primera_fecha",
    "ultima_fecha",
    "numero_filas",
    "fechas_duplicadas",
    "valores_faltantes_totales"
])

display(resumen)

,valor
ticker,EURUSD=X
primera_fecha,2004-01-01
ultima_fecha,2026-07-14
numero_filas,5845
fechas_duplicadas,0
valores_faltantes_totales,0


In [ ]:
# guardo el dataset en un csv

archivo_csv = ruta_raw / "eurusd_yfinance_diario.csv"

datos.to_csv(
    archivo_csv,
    index=True,
    encoding="utf-8"
)

print("Archivo guardado en:")
print(archivo_csv)

Archivo guardado en:
C:\TFM_EURUSD\data\raw\eurusd_yfinance_diario.csv


In [ ]:
# informacion sobre la descarga de los datos

metadatos = {
    "activo": "EUR/USD",
    "ticker": TICKER,
    "fuente": "Yahoo Finance mediante yfinance",
    "frecuencia": INTERVALO,
    "fecha_inicio_solicitada": FECHA_INICIO,
    "fecha_fin_exclusiva": FECHA_FIN,
    "primera_fecha_obtenida": str(datos.index.min().date()),
    "ultima_fecha_obtenida": str(datos.index.max().date()),
    "numero_filas": len(datos),
    "columnas": list(datos.columns),
    "fecha_descarga_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "version_yfinance": yf.__version__,
    "version_pandas": pd.__version__
}

archivo_metadatos = ruta_raw / "eurusd_yfinance_metadatos.json"

with open(archivo_metadatos, "w", encoding="utf-8") as archivo:
    json.dump(
        metadatos,
        archivo,
        ensure_ascii=False,
        indent=4
    )

print("Metadatos guardados en:")
print(archivo_metadatos)

Metadatos guardados en:
C:\TFM_EURUSD\data\raw\eurusd_yfinance_metadatos.json


In [12]:
# SOlo para comproibbar que se puede leer bien el csv

datos_verificacion = pd.read_csv(
    archivo_csv,
    parse_dates=["Date"],
    index_col="Date"
)

if len(datos_verificacion) != len(datos):
    raise ValueError("El número de filas cambió al guardar el archivo.")

if list(datos_verificacion.columns) != list(datos.columns):
    raise ValueError("Las columnas cambiaron al guardar el archivo.")

print("El archivo se guardó y se recuperó correctamente.")
print("Filas verificadas:", len(datos_verificacion))

display(datos_verificacion.tail())

El archivo se guardó y se recuperó correctamente.
Filas verificadas: 5845


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-07-08,1.140381,1.140381,1.143249,1.139186,1.140095,0
2026-07-09,1.142204,1.142204,1.145082,1.142126,1.142191,0
2026-07-10,1.143341,1.143341,1.146263,1.141839,1.143380,0
2026-07-13,1.140446,1.140446,1.144571,1.138680,1.140368,0
2026-07-14,1.138433,1.138433,1.146132,1.137877,1.138369,0
